
# Exercise AI in Network Security (NIDS Robustness Lab)

**What you'll do**
1. Load & preprocess **NSL‑KDD** (official Train+ / Test+ split).  
2. Train a small **MLP** on NSL‑KDD Train+ and evaluate on Test+.  
3. Generate **FGSM** and **PGD** adversarial examples with **ART** and measure robustness.  
4. Load a **sample of CICIDS2017** (50k rows by default).  
5. Build a *second* pipeline & MLP for CICIDS2017 (schema differs), evaluate clean + adversarial.  
6. Compare results across datasets in a simple summary table.

> NSL‑KDD and CICIDS2017 have different feature schemas. We therefore **train & evaluate separate models** per dataset, using dataset‑appropriate preprocessing. The comparison focuses on *relative* accuracy and robustness, not on a shared feature space.

**The goal of this labsession is to understand why it is diffucult to train an NIDS to classify intrusions from benign traffic.**


## Background and acronyms used

NSL-KDD is a benchmark dataset for network intrusion detection — widely used to evaluate anomaly-detection and classification models.

MLP is a type of feed-forward neural network — one of the simplest deep learning architectures.
It consists of layers of neurons: an input layer, one or more hidden layers, and an output layer.
Each neuron applies a weighted sum of its inputs and a nonlinear activation function (like ReLU or tanh).
During training, the model adjusts its weights using backpropagation to minimize prediction error.
In your lab, the MLP will:
- Take the numerical features from the network traffic data as input.
- Learn to classify each connection as normal or attack (and possibly into specific attack types).


FGSM(Fast Gradient sign method) is an adversarial attack method introduced by Goodfellow et al. (2015). https://arxiv.org/pdf/1412.6572v3

PGD(Projected gradient descent) is a stronger iterative version of FGSM.
https://arxiv.org/pdf/1706.06083

## 1) Setup

In [11]:

# Before run this notebook, install the required packages:
# pip -q install tensorflow>=2.17,<2.20 adversarial-robustness-toolbox scikit-learn pandas matplotlib numpy>=1.26,<2.0

import os, sys, math, json, random
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("TensorFlow:", tf.__version__)


TensorFlow: 2.19.1


## 2) Helper functions

In [12]:

def build_mlp(input_dim, n_classes=2, width=128, dropout=0.2):
    model = Sequential([
        Dense(width, activation="relu", input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(dropout),
        Dense(width//2, activation="relu"),
        BatchNormalization(),
        Dropout(dropout),
        Dense(n_classes, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def art_wrap_keras(model, nb_features, nb_classes=2, clip_values=(0.0, 1.0)):
    # ART expects logits if using loss="categorical_crossentropy", but we keep sparse CE with softmax
    classifier = TensorFlowV2Classifier(
        model=model,
        nb_classes=nb_classes,
        input_shape=(nb_features,),
        loss_object=tf.keras.losses.SparseCategoricalCrossentropy(),
        clip_values=clip_values,
        channels_first=False
    )
    return classifier

def evaluate_and_report(model, X_test, y_test, title=""):
    proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(proba, axis=1)
    acc = accuracy_score(y_test, y_pred)
    print(f"{title} accuracy: {acc*100:.2f}%")
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, digits=4))
    return acc

def fgsm_attack(classifier, X, eps=0.1, batch_size=256):
    attack = FastGradientMethod(estimator=classifier, eps=eps, batch_size=batch_size)
    X_adv = attack.generate(X.astype(np.float32))
    return X_adv

def pgd_attack(classifier, X, eps=0.1, eps_step=0.01, max_iter=40, batch_size=256):
    attack = ProjectedGradientDescent(
        estimator=classifier,
        eps=eps,
        eps_step=eps_step,
        max_iter=max_iter,
        batch_size=batch_size
    )
    X_adv = attack.generate(X.astype(np.float32))
    return X_adv

def summarize_results(rows, title="Summary"):
    df = pd.DataFrame(rows, columns=["Dataset","Split/Scope","Clean Acc","FGSM Acc (eps=0.1)","PGD Acc (eps=0.1)"])
    return df


## 3) NSL‑KDD: load, preprocess, train, evaluate, attack

In this section, the NSL-KDD dataset is used to train and evaluate a baseline neural-network classifier for intrusion detection.  
The script first loads the official **Train+** and **Test+** splits from the DEFCON mirror, which contain labeled network connection records describing normal and attack behaviors. Each record includes 41 features such as connection duration, protocol type, service, and error rates.

Categorical attributes (e.g., protocol type, service, flag) are one-hot encoded, while numerical features are standardized using a `StandardScaler` to ensure consistent feature scaling. The preprocessed training and testing sets are then fed into a small **Multilayer Perceptron (MLP)** model, which is trained for several epochs and evaluated on the clean NSL-KDD Test+ data. 

After establishing the clean baseline accuracy, the trained model is wrapped using the **Adversarial Robustness Toolbox (ART)** to generate adversarial examples.  
Two gradient-based attacks are applied:
- **FGSM (Fast Gradient Sign Method)** — a single-step perturbation method introduced by Goodfellow et al. (2015) that adds small, directed noise to each input feature.
- **PGD (Projected Gradient Descent)** — a stronger iterative variant, which performs multiple gradient updates within a bounded perturbation region.

The model’s performance is then re-evaluated on the adversarially perturbed test data to assess its robustness against these attacks.


In [13]:

# Official NSL-KDD Train+/Test+ from defcom17 mirror
TRAIN_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt"
TEST_URL  = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt"

cols = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment",
    "urgent","hot","num_failed_logins","logged_in","num_compromised","root_shell","su_attempted",
    "num_root","num_file_creations","num_shells","num_access_files","num_outbound_cmds",
    "is_host_login","is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate",
    "dst_host_count","dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

def load_nsl(url):
    df = pd.read_csv(url, names=cols, header=None)
    # Binary intrusion labels: normal traffic is 0, every attack is 1.
    labels = df["label"].astype(str).str.strip().str.lower().str.rstrip(".")
    df["label"] = (labels != "normal").astype(int)
    return df

print("Loading NSL-KDD Train+ and Test+ ...")
df_tr = load_nsl(TRAIN_URL)
df_te = load_nsl(TEST_URL)
print("Train:", df_tr.shape, "Test:", df_te.shape)

cat_cols = ["protocol_type","service","flag"]
num_cols = [c for c in df_tr.columns if c not in cat_cols + ["label","difficulty"]]

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
scaler = StandardScaler()

pre = ColumnTransformer([
    ("cat", ohe, cat_cols),
    ("num", scaler, num_cols)
], remainder="drop")

# Train
X_tr = pre.fit_transform(df_tr.drop(columns=["label","difficulty"]))
y_tr = df_tr["label"].values.astype(int)
# Test
X_te = pre.transform(df_te.drop(columns=["label","difficulty"]))
y_te = df_te["label"].values.astype(int)

print("NSL-KDD shapes: X_tr", X_tr.shape, "X_te", X_te.shape)

clip_values = (np.min(X_tr).astype(float), np.max(X_tr).astype(float))

model_nsl = build_mlp(input_dim=X_tr.shape[1], width=128, dropout=0.2)
hist = model_nsl.fit(X_tr, y_tr, epochs=8, batch_size=256, validation_split=0.1, verbose=0)

print("\nClean evaluation on NSL-KDD Test+:")
acc_nsl_clean = evaluate_and_report(model_nsl, X_te, y_te, title="NSL-KDD Test+")

clf_nsl = art_wrap_keras(model_nsl, nb_features=X_tr.shape[1], nb_classes=2, clip_values=clip_values)

print("\nFGSM on NSL-KDD Test+")
X_te_fgsm = fgsm_attack(clf_nsl, X_te, eps=0.1)
acc_nsl_fgsm = evaluate_and_report(model_nsl, X_te_fgsm, y_te, title="NSL-KDD Test+ (FGSM)")

print("\nPGD on NSL-KDD Test+")
X_te_pgd = pgd_attack(clf_nsl, X_te, eps=0.1, eps_step=0.01, max_iter=40)
acc_nsl_pgd = evaluate_and_report(model_nsl, X_te_pgd, y_te, title="NSL-KDD Test+ (PGD)")


Loading NSL-KDD Train+ and Test+ ...
Train: (125973, 43) Test: (22544, 43)
NSL-KDD shapes: X_tr (125973, 122) X_te (22544, 122)


/Users/jonasson/.pyenv/versions/3.11.14/envs/ai_security/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Clean evaluation on NSL-KDD Test+:
NSL-KDD Test+ accuracy: 77.17%

Classification report:
              precision    recall  f1-score   support

           0     0.6698    0.9271    0.7777      9711
           1     0.9222    0.6542    0.7654     12833

    accuracy                         0.7717     22544
   macro avg     0.7960    0.7906    0.7716     22544
weighted avg     0.8135    0.7717    0.7707     22544


FGSM on NSL-KDD Test+
NSL-KDD Test+ (FGSM) accuracy: 27.23%

Classification report:
              precision    recall  f1-score   support

           0     0.1038    0.0903    0.0966      9711
           1     0.3733    0.4100    0.3908     12833

    accuracy                         0.2723     22544
   macro avg     0.2386    0.2502    0.2437     22544
weighted avg     0.2572    0.2723    0.2641     22544


PGD on NSL-KDD Test+


                                      2026-11-03 10:42:45.863067: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


NSL-KDD Test+ (PGD) accuracy: 21.89%

Classification report:
              precision    recall  f1-score   support

           0     0.0793    0.0766    0.0779      9711
           1     0.3185    0.3266    0.3225     12833

    accuracy                         0.2189     22544
   macro avg     0.1989    0.2016    0.2002     22544
weighted avg     0.2155    0.2189    0.2171     22544



### Discussion — Understanding NSL-KDD Results

You have now trained a baseline MLP on the NSL-KDD dataset and evaluated it on both clean and adversarially perturbed data (FGSM and PGD).

Reflect on the following:

1. **Baseline performance**  
   - How accurate is your model on clean NSL-KDD Test+ data? *- it is around 77% accurate on the clean data - this is not super good, but good enough to show it has found some relevant feature signals.*
   - Which attack categories (if you inspected labels) seem hardest to classify, and why? - *idk, TODO*

2. **Adversarial impact**  
   - Compare the accuracy drop between FGSM and PGD attacks. *- PGD is more succesfull and shows a larger drop, but they both succeed in decreasing the performance all the way to 20-30%*
   - Why does PGD typically cause a larger degradation than FGSM? *- PGD does an iterative smaller-step trying to increase the loss in multiple small steps while FGSM only does a single step - PGD is therefore better at finding a better path towards a loss maximisation*
   



## 4) CICIDS2017: load sample, preprocess, train, evaluate, attack
This section extends the analysis to the more recent **CICIDS2017** dataset, which captures realistic network traffic over several days and includes both benign and malicious flows such as DoS, DDoS, PortScan, and Brute Force attacks. The dataset provides over 80 flow-based statistical features per connection, offering a richer representation than NSL-KDD.

The script begins by locating and loading one of the CICIDS2017 CSV files (in this case, the *Friday-WorkingHours-Afternoon-DDoS.pcap_ISCX.csv* capture). Non-numeric columns and missing values are cleaned, extreme feature values are clipped to remove outliers, and a random sample of up to 50,000 flows is extracted for efficiency. A binary label is created, where **0** represents *benign* traffic and **1** denotes *malicious* activity.

After preprocessing, the data is divided into training and test subsets (80/20 split) and standardized using a `StandardScaler`. A small **Multilayer Perceptron (MLP)** model is then trained to classify the network flows and evaluated on clean test data to establish baseline accuracy.

As with the NSL-KDD experiment, the trained classifier is wrapped with the **Adversarial Robustness Toolbox (ART)** and subjected to two adversarial attacks:
- **FGSM (Fast Gradient Sign Method)** — a single-step gradient-based perturbation that slightly modifies input features in the direction of the loss gradient \cite{goodfellow2015explaining}.
- **PGD (Projected Gradient Descent)** — an iterative extension of FGSM that applies multiple small perturbations within a bounded range to generate stronger adversarial examples \cite{madry2017towards}.

The resulting model accuracies under both attacks provide an empirical measure of the model’s robustness to adversarial manipulations in a modern, high-dimensional network dataset.


In [17]:

# Set this to your CICIDS2017 CSV path (or leave None to auto-search current dir)
CIC_PATH = "/Users/jonasson/Documents/MSc Computer Science/ComputerScienceMasters/AISecurity/exercise_week_10/MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"

def find_cic_csv():
    if CIC_PATH and Path(CIC_PATH).exists():
        return Path(CIC_PATH)
    for p in Path(".").glob("*.csv"):
        # Heuristic: CIC files often contain these tokens
        if any(tok in p.name.lower() for tok in ["friday","monday","tuesday","wednesday","thursday","pcap_iscx"]):
            return p
    return None

cic_file = find_cic_csv()
if cic_file is None:
    print("No CICIDS2017 CSV found. Please set CIC_PATH to your file path and re-run this cell.")
else:
    print("Found CIC file:", cic_file)

if cic_file is not None:
    df_cic = pd.read_csv(cic_file)
    df_cic.columns = df_cic.columns.str.strip() 
    print("CICIDS2017 raw shape:", df_cic.shape)
    # Standardize label column name
    label_col = "Label" if "Label" in df_cic.columns else ("label" if "label" in df_cic.columns else None)
    if label_col is None:
        raise ValueError("Could not find 'Label' column in CIC file. Inspect columns: " + str(df_cic.columns[:20].tolist()))
    # Binary label: BENIGN traffic is 0, attack traffic is 1.
    y_cic = (df_cic[label_col].astype(str).str.strip().str.upper() != "BENIGN").astype(int)

    # Drop non-numeric columns except label
    df_num = df_cic.drop(columns=[label_col], errors="ignore").select_dtypes(include=[np.number]).copy()

    # Clean inf/NaN
    df_num = df_num.replace([np.inf, -np.inf], np.nan).fillna(0)

    # Optional: clip extreme values to reasonable range to stabilize training
    df_num = df_num.clip(df_num.quantile(0.001), df_num.quantile(0.999), axis=1)

    # Sample
    N_SAMPLE = 50000 if len(df_num) > 50000 else len(df_num)
    samp = np.random.RandomState(42).choice(len(df_num), size=N_SAMPLE, replace=False)
    X_cic = df_num.iloc[samp].values.astype(np.float32)
    y_cic = y_cic.iloc[samp].values.astype(int)
    print("CIC sample:", X_cic.shape)

    # Train/val/test split (we'll keep it within CIC dataset)
    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_cic, y_cic, test_size=0.2, random_state=RANDOM_SEED, stratify=y_cic)

    # Scale using statistics from the training split only.
    scaler_cic = StandardScaler()
    Xc_tr = scaler_cic.fit_transform(Xc_tr)
    Xc_te = scaler_cic.transform(Xc_te)

    # Model
    model_cic = build_mlp(input_dim=Xc_tr.shape[1], width=128, dropout=0.2)
    model_cic.fit(Xc_tr, yc_tr, epochs=6, batch_size=256, validation_split=0.1, verbose=0)

    print("\nClean evaluation on CICIDS2017 sample (held-out 20%):")
    acc_cic_clean = evaluate_and_report(model_cic, Xc_te, yc_te, title="CICIDS2017 (clean)")

    # ART wrapper & attacks
    cic_clip = (float(np.min(Xc_tr)), float(np.max(Xc_tr)))
    clf_cic = art_wrap_keras(model_cic, nb_features=Xc_tr.shape[1], nb_classes=2, clip_values=cic_clip)

    print("\nFGSM on CICIDS2017 (eps=0.1)...")
    Xc_te_fgsm = fgsm_attack(clf_cic, Xc_te, eps=0.1)
    acc_cic_fgsm = evaluate_and_report(model_cic, Xc_te_fgsm, yc_te, title="CICIDS2017 (FGSM)")

    print("\nPGD on CICIDS2017 (eps=0.1)...")
    Xc_te_pgd = pgd_attack(clf_cic, Xc_te, eps=0.1, eps_step=0.01, max_iter=40)
    acc_cic_pgd = evaluate_and_report(model_cic, Xc_te_pgd, yc_te, title="CICIDS2017 (PGD)")


Found CIC file: /Users/jonasson/Documents/MSc Computer Science/ComputerScienceMasters/AISecurity/exercise_week_10/MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
CICIDS2017 raw shape: (225745, 79)
CIC sample: (50000, 78)


/Users/jonasson/.pyenv/versions/3.11.14/envs/ai_security/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Clean evaluation on CICIDS2017 sample (held-out 20%):
CICIDS2017 (clean) accuracy: 99.92%

Classification report:
              precision    recall  f1-score   support

           0     0.9986    0.9995    0.9991      4299
           1     0.9996    0.9989    0.9993      5701

    accuracy                         0.9992     10000
   macro avg     0.9991    0.9992    0.9992     10000
weighted avg     0.9992    0.9992    0.9992     10000


FGSM on CICIDS2017 (eps=0.1)...
CICIDS2017 (FGSM) accuracy: 87.27%

Classification report:
              precision    recall  f1-score   support

           0     0.7862    0.9667    0.8672      4299
           1     0.9697    0.8018    0.8778      5701

    accuracy                         0.8727     10000
   macro avg     0.8779    0.8843    0.8725     10000
weighted avg     0.8908    0.8727    0.8732     10000


PGD on CICIDS2017 (eps=0.1)...


                                      2026-11-03 11:05:59.377190: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


CICIDS2017 (PGD) accuracy: 81.03%

Classification report:
              precision    recall  f1-score   support

           0     0.7063    0.9565    0.8126      4299
           1     0.9552    0.7001    0.8080      5701

    accuracy                         0.8103     10000
   macro avg     0.8308    0.8283    0.8103     10000
weighted avg     0.8482    0.8103    0.8100     10000



### Discussion — Generalisation and Robustness on CICIDS2017

CICIDS2017 represents modern, high-dimensional network flows. Compare your results here to those from NSL-KDD.

1. **Dataset complexity**  
   - How does the model’s baseline accuracy differ between CICIDS2017 and NSL-KDD?  
   - What might explain this difference (feature richness, traffic diversity, data imbalance)?

2. **Cross-dataset reasoning**  
   - Did your model generalise similarly well on both datasets?  
   - What does that suggest about the model’s ability to recognise *semantic* attack behaviour rather than dataset-specific patterns?

3. **Robustness patterns**  
   - Is the adversarial accuracy drop (FGSM/PGD) larger or smaller than on NSL-KDD?  
   - Why might higher-dimensional data be either easier or harder to attack?



## 5) Summary & comparison

In [18]:

rows = []
rows.append(["NSL-KDD","Test+", f"{acc_nsl_clean:.3f}", f"{acc_nsl_fgsm:.3f}", f"{acc_nsl_pgd:.3f}"])
rows.append(["CICIDS2017","Held-out 20%", f"{acc_cic_clean:.3f}", f"{acc_cic_fgsm:.3f}", f"{acc_cic_pgd:.3f}"])
summary_df = summarize_results(rows, title="Summary")
summary_df


,Dataset,Split/Scope,Clean Acc,FGSM Acc (eps=0.1),PGD Acc (eps=0.1)
0,NSL-KDD,Test+,0.772,0.272,0.219
1,CICIDS2017,Held-out 20%,0.999,0.873,0.810


### Reflection — AI Robustness in Network Security

Combine your findings from both datasets.

1. In what ways did adversarial examples change your view of “accuracy” as a metric for NIDS reliability?  
2. Why is adversarial robustness a particularly critical property for systems defending real networks?  
3. Do you think epsilon-ball robustness a good and sufficient measurement for NIDS?
